In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: 2.0-dev.

In [2]:
import numpy as np
from aiida import engine, orm
from aiida.orm import Dict, KpointsData, StructureData, load_group
from aiida.tools import get_explicit_kpoints_path
from ase.build import bulk

###
# set up code
computer = orm.load_computer("localhost")

In [3]:
code = load_code("abacus-3.10@localhost")
builder = code.get_builder()

builder.metadata.options = {
    "resources": {
        "num_machines": 1,
        "num_mpiprocs_per_machine": 1,  # use 1 cores per machine
    },
    "max_wallclock_seconds": 180,  # how long it can run before it should be killed
    # 'withmpi': False, # Set withmpi to False in case abacus was compiled without MPI support.
}

In [4]:
get_explicit_kpoints_path

<function aiida.tools.data.array.kpoints.main.get_explicit_kpoints_path(structure, method='seekpath', **kwargs)>

In [5]:
structure = StructureData(ase=bulk("Si", "fcc", 5.43))
seekpathout = get_explicit_kpoints_path(structure, reference_distance=0.15)
builder.structure = seekpathout["primitive_structure"]

/home/bonan/miniconda3/envs/aiida-2.4-dev/lib/python3.12/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['std_lattice']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(
/home/bonan/miniconda3/envs/aiida-2.4-dev/lib/python3.12/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['std_positions']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(
/home/bonan/miniconda3/envs/aiida-2.4-dev/lib/python3.12/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['std_types']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(
/home/bonan/miniconda3/envs/aiida-2.4-dev/lib/python3.12/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['number']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instea

In [6]:
builder.kpoints = KpointsData()
builder.kpoints.set_kpoints_mesh([6, 6, 4], offset=[0.5, 0.5, 0.5])

In [7]:
pseudo_family = load_group("PseudoDojo/0.4/PBE/SR/standard/upf")
builder.pseudos = pseudo_family.get_pseudos(structure=structure)

In [8]:
builder.parameters = {
    "input": {  # pseudo_dir will be set by the plugin based on the pseudos
        # 'symmetry': 1,
        "basis_type": "pw",
        "ecutwfc": 100,
        "scf_thr": 1e-4,  # 1e-7,
        # 'scf_nmax': 100,
        "device": "cpu",
        # 'ks_solver': 'dav_subspace',
        # 'precision': 'double',
    }
}

In [9]:
node = load_node(4090)

# NSCF

In [10]:
builder.parameters = {
    "input": {  # pseudo_dir will be set by the plugin based on the pseudos
        # 'symmetry': 1,
        "basis_type": "pw",
        "ecutwfc": 100,
        "scf_thr": 1e-4,  # 1e-7,
        # 'scf_nmax': 100,
        "device": "cpu",
        # 'ks_solver': 'dav_subspace',
        # 'precision': 'double',
        "calculation": "nscf",
    }
}

In [11]:
builder.kpoints = seekpathout["explicit_kpoints"]
builder.settings = {"include_bands": True}
builder.restart_folder = node.outputs.remote_folder

In [12]:
nscf_results, nscf_node = engine.run.get_node(builder)

05/13/2025 04:12:52 PM <331613> aiida.broker.rabbitmq: [WARNING] RabbitMQ v3.12.1 is not supported and will cause unexpected problems!
05/13/2025 04:12:52 PM <331613> aiida.broker.rabbitmq: [WARNING] It can cause long-running workflows to crash and jobs to be submitted multiple times.
05/13/2025 04:12:52 PM <331613> aiida.broker.rabbitmq: [WARNING] See https://github.com/aiidateam/aiida-core/wiki/RabbitMQ-version-to-use for details.
05/13/2025 04:13:04 PM <331613> aiida.parser.AbacusParser: [WARNING] The following expected files are missing: ['OUT.aiida/device.log']


In [13]:
nscs

SyntaxError: invalid syntax (4209967521.py, line 1)